# 🧬 AMR-Predictor-ML: Predicting Meropenem Resistance in *Klebsiella pneumoniae*
## Phase 4: Feature Selection & Dimensionality Reduction

### 📋 Phase Overview
Our baseline models (Random Forest and XGBoost) suffered from the **Curse of Dimensionality** ($4,096$ features vs. $2,268$ training samples), leading to a low Recall score (~49%) for the Resistant class. In this phase, we will apply statistical feature selection to filter out the noise and retain only the most informative K-mers.

### 🎯 Method: ANOVA F-Test (SelectKBest)
We will use the **Analysis of Variance (ANOVA) F-test** via scikit-learn's `SelectKBest`. This method evaluates each K-mer independently, calculating an F-statistic to score how strongly its frequency varies between the 'Susceptible' and 'Resistant' phenotypic groups. We will select the **Top 500 most significant K-mers** to rebuild our dataset.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif

# 1. Load the golden dataset
dataset_path = "../data/processed/kmers_6_dataset.csv"
print("⏳ Loading Golden Dataset...")
df = pd.read_csv(dataset_path)

# 2. Separate Target, Metadata, and Features
metadata_cols = ['Genome ID', 'Genome Name', 'Antibiotic', 'Resistant Phenotype', 'AMR_Label']
metadata_df = df[metadata_cols].copy() # Keep metadata safe aside

y = df['AMR_Label'].values
X = df.drop(columns=metadata_cols)
feature_names = X.columns.tolist()

print(f"📊 Original Shape: {X.shape[0]} samples × {X.shape[1]} features")

# 3. Apply SelectKBest with ANOVA F-Test to select top 500 features
# k=500 is an optimal starting point to reduce dimensionality by ~88%
num_features_to_select = 500
print(f"✂️  Selecting Top {num_features_to_select} most statistically significant K-mers...")
selector = SelectKBest(score_func=f_classif, k=num_features_to_select)

X_selected = selector.fit_transform(X, y)

# 4. Get the names of the selected K-mers
selected_indices = selector.get_support(indices=True)
selected_kmer_names = [feature_names[i] for i in selected_indices]

print("✅ Feature Selection Complete!")
print(f"📐 New Feature Matrix Shape: {X_selected.shape}")

# 5. Let's see the top 5 highest-scoring K-mers
scores = selector.scores_
kmer_scores = pd.DataFrame({'K-mer': feature_names, 'F-Score': scores})
top_kmers = kmer_scores.sort_values(by='F-Score', ascending=False).head(5)

print("\n🔝 Top 5 K-mers with the highest biological correlation to resistance:")
print(top_kmers.to_string(index=False))

⏳ Loading Golden Dataset...
📊 Original Shape: 2836 samples × 4096 features
✂️  Selecting Top 500 most statistically significant K-mers...
✅ Feature Selection Complete!
📐 New Feature Matrix Shape: (2836, 500)

🔝 Top 5 K-mers with the highest biological correlation to resistance:
 K-mer   F-Score
CCTAGG 22.571596
CCCTAG 21.382189
CCTAGA 14.614120
ACCTAG  9.848678
GGGGGG  8.739710


In [2]:
# 1. Convert the selected numpy matrix back into a structured Pandas DataFrame
df_selected_features = pd.DataFrame(X_selected, columns=selected_kmer_names)

# 2. Reset indices to guarantee perfect alignment when concatenating
metadata_df.reset_index(drop=True, inplace=True)
df_selected_features.reset_index(drop=True, inplace=True)

# 3. Concatenate metadata and selected genomic features side-by-side
df_selected_final = pd.concat([metadata_df, df_selected_features], axis=1)

# 4. Save this new lightweight dataset to data/processed/
output_selected_path = "../data/processed/kmers_6_selected_500.csv"
df_selected_final.to_csv(output_selected_path, index=False)

print(f"🎉 Success! Reduced Dataset saved to: {output_selected_path}")
print(f"📐 Final Matrix Shape: {df_selected_final.shape[0]} samples × {df_selected_final.shape[1]} columns")

🎉 Success! Reduced Dataset saved to: ../data/processed/kmers_6_selected_500.csv
📐 Final Matrix Shape: 2836 samples × 505 columns


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

# 1. Load the newly reduced dataset
dataset_path = "../data/processed/kmers_6_selected_500.csv"
print("⏳ Loading Reduced Selected Dataset...")
df_selected = pd.read_csv(dataset_path)

# 2. Separate Features and Target
metadata_cols = ['Genome ID', 'Genome Name', 'Antibiotic', 'Resistant Phenotype', 'AMR_Label']
y = df_selected['AMR_Label'].values
X = df_selected.drop(columns=metadata_cols).values

# 3. Stratified Train/Test Split (80/20) - Same exact split parameters as before
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"🏋️ Training Set Shape: {X_train.shape}")
print(f"🧪 Testing Set Shape : {X_test.shape}\n")

# ==========================================
# 🔥 Model 1: Random Forest on Selected Features
# ==========================================
print("🌲 Training Random Forest on Top 500 K-mers...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print("\n📊 --- Random Forest (After Feature Selection) ---")
print(f"🎯 Accuracy : {accuracy_score(y_test, y_pred_rf)*100:.2f}%")
print(f"📈 ROC-AUC  : {roc_auc_score(y_test, y_prob_rf)*100:.2f}%")
print(classification_report(y_test, y_pred_rf, target_names=["Susceptible (0)", "Resistant (1)"]))

# ==========================================
# 🔥 Model 2: XGBoost on Selected Features
# ==========================================
print("\n🚀 Training XGBoost on Top 500 K-mers...")
xgb_model = xgb.XGBClassifier(
    n_estimators=200, learning_rate=0.1, max_depth=6, random_state=42, n_jobs=-1, eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("\n📊 --- XGBoost (After Feature Selection) ---")
print(f"🎯 Accuracy : {accuracy_score(y_test, y_pred_xgb)*100:.2f}%")
print(f"📈 ROC-AUC  : {roc_auc_score(y_test, y_prob_xgb)*100:.2f}%")
print(classification_report(y_test, y_pred_xgb, target_names=["Susceptible (0)", "Resistant (1)"]))

⏳ Loading Reduced Selected Dataset...
🏋️ Training Set Shape: (2268, 500)
🧪 Testing Set Shape : (568, 500)

🌲 Training Random Forest on Top 500 K-mers...

📊 --- Random Forest (After Feature Selection) ---
🎯 Accuracy : 66.90%
📈 ROC-AUC  : 69.71%
                 precision    recall  f1-score   support

Susceptible (0)       0.70      0.82      0.76       353
  Resistant (1)       0.59      0.41      0.49       215

       accuracy                           0.67       568
      macro avg       0.64      0.62      0.62       568
   weighted avg       0.66      0.67      0.65       568


🚀 Training XGBoost on Top 500 K-mers...

📊 --- XGBoost (After Feature Selection) ---
🎯 Accuracy : 68.49%
📈 ROC-AUC  : 70.38%
                 precision    recall  f1-score   support

Susceptible (0)       0.73      0.79      0.76       353
  Resistant (1)       0.60      0.52      0.55       215

       accuracy                           0.68       568
      macro avg       0.66      0.65      0.66       56